# Lab 4. VQE — Preparing a Ground State with a Circuit and Checking the Result

## Introduction

In this lab, <strong>you will calculate the energy of a state prepared by a quantum circuit and vary the circuit angles to search for the lowest-energy state.</strong> After the search, you will examine both the energy and the prepared state, then decide what to change if the result is not good enough.

In Lab 3, you recovered stored logical information after an error. Here, you will use energy to evaluate states as you search for a desired state to prepare. The operator that represents the energy of a quantum system is called the <strong>Hamiltonian</strong>. We will write it as $\mathcal H$ to distinguish it from the Hadamard gate $H$.

For a state $|\psi\rangle$, the energy expectation value is

$$
E=\langle\psi|\mathcal H|\psi\rangle
$$

This is the average obtained by repeatedly preparing the same state and measuring its energy. It does not mean that every measurement of a general state gives the same result. In this lab, “the energy of a state” means this expectation value.

An eigenstate $|E_k\rangle$ of the Hamiltonian satisfies $\mathcal H|E_k\rangle=E_k|E_k\rangle$, and an energy measurement of this state gives $E_k$. The smallest possible eigenvalue is the <strong>ground energy</strong> $E_0$, and a state with that energy is called a <strong>ground state</strong>.

Writing a normalized state as a superposition of energy eigenstates gives

$$
|\psi\rangle=\sum_k c_k|E_k\rangle,
\qquad \sum_k|c_k|^2=1,
\qquad E=\sum_k|c_k|^2E_k\ge E_0.
$$

The energy is a probability-weighted average of the eigenvalues, so it cannot be smaller than the smallest eigenvalue. This is the <strong>variational principle</strong> used in this lab. You will lower the energy to search for the ground state, then check how close the resulting energy and state are to the target.

To try different states, you will use a circuit with adjustable rotation angles. A fixed circuit structure whose parameters specify the states to explore is called an <strong>ansatz</strong>. <strong>The circuit structure determines the set of states that can be prepared, and the angles select one state from that set.</strong> Adjustable angles do not necessarily let a circuit prepare every possible state.

A classical optimizer repeatedly calls an <strong>objective function</strong> that takes angles as input and returns one energy value. The overall process is <strong>choose angles → prepare a state → evaluate its energy → update the angles → check the result</strong>. This combination of a parameterized quantum circuit and classical optimization to find a ground state is called <strong>VQE (Variational Quantum Eigensolver)</strong>. Each new set of angles starts with a fresh state preparation; you do not keep adding gates to a state after it has been measured.

The two-qubit Hamiltonian and reference energy for this lab are

$$
\mathcal H=-Z_0Z_1-X_0Y_1-\frac12(Z_0+Z_1),
\qquad E_0=-1-\sqrt2\approx-2.4142.
$$

Subscripts identify the qubits on which the operators act. Problem 1 explains how to measure each term and combine the results into an energy. This problem is small enough to diagonalize its matrix directly, so the exact ground energy is provided as a reference. The optimizer is not given the correct angles or the ground state; it uses the energy returned by your objective function.

In the review material, you ran VQE with a circuit built from $R_y$ and CX and compared it with a version without CX. Here, you will start with that familiar circuit and ask <strong>whether it can also prepare the ground state of the new Hamiltonian</strong>. If necessary, you will extend the circuit and explain the improvement using the state it prepares.

| Problem | What you will investigate |
|---|---|
| <strong>1. Calculating energy from measurements</strong> | Interpret each observable and combine expectation values into one energy. |
| <strong>2. Finding the states a circuit can explore</strong> | Use the first circuit's state and energy expressions to determine what it can achieve. |
| <strong>3. Extending and optimizing the circuit</strong> | Modify the circuit to prepare the needed states and connect it to the objective function. |
| <strong>4. Checking results and diagnosing problems</strong> | Distinguish termination messages, exact energy, state fidelity, and finite-shot results. |

At the end, you will change the sign of one Hamiltonian term and decide how to reuse the circuit and its parameters. <strong>An optimizer stopping and a ground state being found are different claims.</strong> You will use the actual results to examine the circuit's expressive range, the search process, and the effect of measurement.

---

## How to Use This Notebook

Each section moves from the purpose of a calculation and definitions of its symbols to the relevant equations, examples, and exercises. As you read, follow how the state prepared by the circuit connects to the calculation that evaluates it.

Run the code cells from top to bottom. The provided code contains tools for states and expectation values, optimization, measurement records, and tables, so those cells must also be run. You do not need to analyze every implementation detail; each problem explains how to use the tools it needs.

1. <strong>Write your calculations and predictions first.</strong> Identify the Hamiltonian, circuit structure, and adjustable angles, then predict the result.
2. <strong>Complete and check the TODOs.</strong> Fill in the area between `TODO BEGIN` and `TODO END`, keeping the function names and parameters unchanged. Run the answer cell, followed by its public check cell.
3. <strong>Enable observations and run the relevant experiment.</strong> In the student notebook, `RUN_OBSERVATIONS` starts as `False`. After writing your prediction, set it to `True` and run the configuration cell and the relevant observation cell. For each new question, predict before opening its results. To disable observations again, set it to `False` and rerun the configuration cell. Public function checks work independently of this setting.
4. <strong>Add your observations and reasoning.</strong> If you edit code, rerun the answer, check, and connected experiment cells. If the circuit changes, recompute earlier optimization results as well.

| Code you will write | Input and output |
|---|---|
| State-preparation circuit | Builds a circuit from the angles. |
| Energy evaluator | Converts observable expectation values into an energy. |
| Optimization objective | Connects state preparation to energy evaluation and returns one energy value. |

The optimizer is provided. Your objective must use the angles it receives on every call, rather than return a known correct energy directly. For the final application, enter your predicted value and proposed angles, then enable the separate `RUN_APPLICATION` setting to check them.

Rotation angles are in <strong>radians</strong>. For two qubits, Qiskit's statevector basis and Pauli strings use $q_1q_0$ order. Refer to the mapping tables when translating qubit subscripts from equations into code.

The first part uses <strong>exact expectation values</strong> of ideal states. Later, you will estimate energy from a fixed number of fresh state preparations and measurements. These repetitions are called <strong>shots</strong>, and finite samples fluctuate. Distinguish exact values from estimates, and check whether a shot count is per measurement setting or the total.

In a check cell, <strong>Incomplete</strong> means a TODO remains, <strong>Failed</strong> means an execution error or an unmet condition, and <strong>Passed</strong> means the public code check passed. Written answers are separate. Use results and reasoning to distinguish observing a lower energy, finding a minimum within a chosen circuit family, and reaching the actual ground state.

<strong>You may use AI to study in this lab.</strong> Use it to explain concepts, check calculations, or debug, but run the suggested code yourself. In written answers, use your actual observations to explain in your own words why the circuit changes and checks are appropriate.

## Preparing the Environment

This notebook was verified with Python 3.12, Qiskit 2.3.1, NumPy 2.2.6, and SciPy 1.18.1. For local use, install the accompanying `requirements.txt` and select that environment as your Jupyter kernel.

If the packages are missing, run the following command in a new code cell, restart the kernel, and run the notebook from the top.

```python
%pip install qiskit==2.3.1 numpy==2.2.6 scipy==1.18.1
```

Run the provided tools cell below as well. Long code may appear collapsed. Each problem explains the tools and how to use them, so you do not need to analyze their entire implementation before starting.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy", "scipy"):
    print(package, version(package))

### Provided Tools and Observation Settings

Run the tools cell below. Public checks support learning; written answers must be completed separately. Write your prediction and calculations for each question before viewing its observation results.

The student notebook starts with `RUN_OBSERVATIONS=False`. When you are ready to observe a result, change the setting below to `True`, run that cell, and run the observation cell for the question. This setting is not a function TODO. The final application has a separate execution setting.

In [ ]:
'Provided tools: states, measurements, optimization, and public checks for two-qubit VQE.'
import numbers
from html import escape
from types import CodeType, FunctionType, ModuleType
from contextlib import contextmanager
import numpy as np
from scipy.optimize import minimize
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.circuit import Gate, Instruction
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector, state_fidelity

ATOL = 1e-9
ENERGY_TOL = 1e-6
FIDELITY_TOL = 1e-6
OBSERVABLE_KEYS = ('ZZ', 'YX', 'IZ', 'ZI')
OBSERVABLES = {key: SparsePauliOp(key) for key in OBSERVABLE_KEYS}
CHECK_LABELS = ('1b · Energy evaluator', '3a · Relative-phase circuit', '3b · Optimization objective')
CHECK_STATUS = {}
LAB_RESULTS = {}


def show_table(headers, rows):
    def cell_text(value):
        if isinstance(value, (float, np.floating)):
            return f'{value:.8f}'
        if isinstance(value, (tuple, list)):
            return '(' + ', '.join(cell_text(x) for x in value) + ')'
        return str(value)
    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError('Check the number of columns in the table.')
    heading = ''.join(f'<th>{escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(cell_text(x))}</td>' for x in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>'))


def _real(value, label='Return value'):
    if not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_)) or not np.isfinite(value):
        raise AssertionError(f'{label}: use a finite real number.')
    return float(value)


def _positive_integer(value, label):
    if not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)) or value < 1:
        raise ValueError(f'{label}: a positive integer is required.')
    return int(value)


def hamiltonian(xy_coeff=-1.0):
    coefficient = _real(xy_coeff, 'XY coefficient')
    return SparsePauliOp.from_list([('ZZ', -1.0), ('YX', coefficient), ('IZ', -0.5), ('ZI', -0.5)])


@contextmanager
def _instrument_dependencies(function, replacements):
    'Temporarily instrument functions and aliases on the actual call path, then restore every reference.'
    if not isinstance(function, FunctionType):
        raise AssertionError('Write your answer inside the provided function definition.')
    namespaces, functions, originals = {}, {}, {}
    library_prefixes = ('numpy', 'scipy', 'qiskit', 'IPython', 'importlib', 'contextlib')

    def code_names(code):
        names = set(code.co_names)
        for constant in code.co_consts:
            if isinstance(constant, CodeType):
                names.update(code_names(constant))
        return names

    def dependency(value, name=None):
        key = name if name in replacements else getattr(value, '__name__', None)
        if isinstance(value, FunctionType) and key in replacements:
            originals[id(value)] = (value, replacements[key])
            return True
        return False

    def visit(candidate):
        if not isinstance(candidate, FunctionType) or id(candidate) in functions:
            return
        if dependency(candidate):
            return
        if (candidate.__module__ or '').startswith(library_prefixes):
            return
        functions[id(candidate)] = candidate
        namespace = candidate.__globals__
        namespaces[id(namespace)] = namespace
        for name in replacements:
            dependency(namespace.get(name), name)
        names = code_names(candidate.__code__)
        for name in names:
            value = namespace.get(name)
            if isinstance(value, FunctionType):
                if not dependency(value, name):
                    visit(value)
            elif isinstance(value, ModuleType) and not value.__name__.startswith(library_prefixes):
                module_namespace = vars(value)
                namespaces[id(module_namespace)] = module_namespace
                for attribute in names:
                    target = module_namespace.get(attribute)
                    if not dependency(target, attribute):
                        visit(target)
        for value in (*tuple(candidate.__defaults__ or ()),
                      *tuple((candidate.__kwdefaults__ or {}).values())):
            if not dependency(value):
                visit(value)
        for cell in candidate.__closure__ or ():
            try:
                value = cell.cell_contents
            except ValueError:
                continue
            if not dependency(value):
                visit(value)

    visit(function)
    saved_bindings, saved_defaults, saved_cells = [], [], []
    seen_cells = set()

    def substituted(value):
        match = originals.get(id(value))
        return match[1] if match is not None and value is match[0] else value

    try:
        for namespace in namespaces.values():
            for name, value in tuple(namespace.items()):
                replacement = substituted(value)
                if replacement is not value:
                    saved_bindings.append((namespace, name, value))
                    namespace[name] = replacement
        for candidate in functions.values():
            defaults, keyword_defaults = candidate.__defaults__, candidate.__kwdefaults__
            new_defaults = None if defaults is None else tuple(substituted(value) for value in defaults)
            new_keywords = None if keyword_defaults is None else {
                name: substituted(value) for name, value in keyword_defaults.items()}
            if (defaults is not None and any(a is not b for a, b in zip(defaults, new_defaults))) or (
                    keyword_defaults is not None and any(keyword_defaults[key] is not new_keywords[key] for key in keyword_defaults)):
                saved_defaults.append((candidate, defaults, keyword_defaults))
                candidate.__defaults__, candidate.__kwdefaults__ = new_defaults, new_keywords
            for cell in candidate.__closure__ or ():
                if id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                try:
                    value = cell.cell_contents
                except ValueError:
                    continue
                replacement = substituted(value)
                if replacement is not value:
                    saved_cells.append((cell, value))
                    cell.cell_contents = replacement
        yield
    finally:
        for cell, value in reversed(saved_cells):
            cell.cell_contents = value
        for candidate, defaults, keyword_defaults in reversed(saved_defaults):
            candidate.__defaults__, candidate.__kwdefaults__ = defaults, keyword_defaults
        for namespace, name, value in reversed(saved_bindings):
            namespace[name] = value


def _check_circuit_operations(circuit, allowed=None, active=None):
    'Check allowed primitive gates and user-defined circuits composed only of those gates.'
    forbidden = {'measure', 'reset', 'initialize', 'set_statevector', 'set_density_matrix',
                 'if_else', 'while_loop', 'for_loop', 'switch_case'}
    active = set() if active is None else active
    if id(circuit) in active:
        raise AssertionError('Recursive composite-gate definitions are not allowed.')
    active.add(id(circuit))
    try:
        if circuit.num_clbits:
            raise AssertionError('Do not include classical bits in a state-preparation circuit.')
        for item in circuit.data:
            operation = item.operation
            if operation.name in forbidden or operation.num_clbits:
                raise AssertionError('Use unitary gates without measurement, initialization, reset, or conditional control flow.')
            if allowed is not None and operation.name in allowed:
                continue
            # Expand only circuits wrapped with to_gate/to_instruction. Do not use
            # decompositions of other primitive gates to bypass the allowed-gate condition.
            if operation.base_class in (Gate, Instruction) and operation.definition is not None:
                _check_circuit_operations(operation.definition, allowed, active)
            elif allowed is not None:
                raise AssertionError('Use the primitive gates allowed for this function. Composite gates are checked internally too.')
    finally:
        active.remove(id(circuit))


def circuit_state(circuit, require_ansatz=False):
    if not isinstance(circuit, QuantumCircuit):
        raise AssertionError('Return a QuantumCircuit.')
    if (circuit.num_qubits, circuit.num_clbits) != (2, 0) or circuit.parameters:
        raise AssertionError('A two-qubit circuit with bound numeric angles and no classical bits is required.')
    allowed = {'ry', 'rz', 'cx', 'barrier'} if require_ansatz else None
    _check_circuit_operations(circuit, allowed)
    matrix = np.asarray(Operator(circuit).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(4), atol=ATOL, rtol=0):
        raise AssertionError('A finite unitary circuit is required.')
    return Statevector.from_instruction(circuit)


def _expectations_from_state(state):
    return {key: float(np.real(state.expectation_value(operator))) for key, operator in OBSERVABLES.items()}


def expectations_from_circuit(circuit):
    'Return exact ZZ, YX, IZ, and ZI expectations for the same circuit state.'
    return _expectations_from_state(circuit_state(circuit))


def exact_energy(circuit, xy_coeff=-1.0):
    'For verification: calculate directly with the Hamiltonian operator.'
    return float(np.real(circuit_state(circuit).expectation_value(hamiltonian(xy_coeff))))


def ground_reference(xy_coeff=-1.0):
    'Exact eigenvalues and ground state used in verification.'
    eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian(xy_coeff).to_matrix())
    return {'energy': float(eigenvalues[0]), 'state': Statevector(eigenvectors[:, 0]),
            'eigenvalues': tuple(map(float, eigenvalues))}


def validate_result(circuit, xy_coeff=-1.0):
    state = circuit_state(circuit)
    reference = ground_reference(xy_coeff)
    energy = float(np.real(state.expectation_value(hamiltonian(xy_coeff))))
    delta = energy - reference['energy']
    fidelity = float(state_fidelity(state, reference['state']))
    return {'energy': energy, 'reference_energy': reference['energy'], 'delta_energy': delta,
            'fidelity': fidelity, 'energy_pass': abs(delta) <= ENERGY_TOL,
            'state_pass': fidelity >= 1 - FIDELITY_TOL}


def real_ansatz(theta):
    circuit = QuantumCircuit(2)
    circuit.ry(_real(theta, 'theta'), 0)
    circuit.cx(0, 1)
    return circuit


def phase_before_rotation(theta, phi):
    'Comparison circuit for 3(c): initial RZ, then RY and CX.'
    circuit = QuantumCircuit(2)
    circuit.rz(_real(phi, 'phi'), 0)
    circuit.ry(_real(theta, 'theta'), 0)
    circuit.cx(0, 1)
    return circuit


def measurement_basis(setting):
    circuit = QuantumCircuit(2)
    if setting == 'XY':
        circuit.h(0)
        circuit.sdg(1)
        circuit.h(1)
    elif setting != 'Z':
        raise ValueError('The measurement setting must be Z or XY.')
    return circuit


def basis_probabilities(circuit, setting):
    state = circuit_state(circuit).evolve(measurement_basis(setting))
    probabilities = np.asarray(state.probabilities(), dtype=float)
    return probabilities / probabilities.sum()


def _counts_vector(counts):
    if not isinstance(counts, dict) or not counts:
        raise ValueError('A dictionary of two-bit strings and integer counts is required.')
    if any(key not in {'00', '01', '10', '11'} for key in counts):
        raise ValueError('Measurement strings contain two bits in q1q0 order.')
    if any(not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0 for n in counts.values()):
        raise ValueError('Measurement counts must be nonnegative integers.')
    vector = np.array([counts.get(f'{i:02b}', 0) for i in range(4)], dtype=float)
    if vector.sum() <= 0:
        raise ValueError('The total measurement count must be positive.')
    return vector / vector.sum()


def terms_from_counts(z_counts, xy_counts):
    'Average the three Z-record statistics and the separate XY record within their respective settings.'
    z_probs = _counts_vector(z_counts)
    xy_probs = _counts_vector(xy_counts)
    q0 = np.array([1, -1, 1, -1])
    q1 = np.array([1, 1, -1, -1])
    parity = q0 * q1
    return {'ZZ': float(z_probs @ parity), 'YX': float(xy_probs @ parity),
            'IZ': float(z_probs @ q0), 'ZI': float(z_probs @ q1)}


def sample_energy(circuit, energy_function, shots, seed=None, xy_coeff=-1.0):
    shots = _positive_integer(shots, 'Shots per setting')
    streams = np.random.SeedSequence(seed).spawn(2)
    records = {}
    for setting, stream in zip(('Z', 'XY'), streams):
        values = np.random.default_rng(stream).multinomial(shots, basis_probabilities(circuit, setting))
        records[setting] = {f'{index:02b}': int(count) for index, count in enumerate(values) if count}
    terms = terms_from_counts(records['Z'], records['XY'])
    energy = _real(energy_function(terms, xy_coeff=xy_coeff), 'Energy estimate')
    return {'energy': energy, 'expectations': terms, 'counts': records,
            'shots_per_setting': shots, 'total_shots': 2 * shots}


def repeated_energy_estimates(circuit, energy_function, shots_values=(64, 256, 1024), repeats=20, seed=20261109, xy_coeff=-1.0):
    repeats = _positive_integer(repeats, 'Number of repetitions')
    if repeats < 2:
        raise ValueError('At least two repetitions are needed for a sample standard deviation.')
    shots_values = tuple(_positive_integer(value, 'Shots per setting') for value in shots_values)
    streams = np.random.SeedSequence(seed).spawn(len(shots_values) * repeats)
    rows = []
    exact = exact_energy(circuit, xy_coeff)
    for group, shots in enumerate(shots_values):
        values = []
        for index in range(repeats):
            local_seed = int(streams[group * repeats + index].generate_state(1)[0])
            trial = sample_energy(circuit, energy_function, shots, local_seed, xy_coeff)
            values.append(trial['energy'])
        rows.append({'shots_per_setting': shots, 'repeats': repeats,
                     'total_shots_per_estimate': 2 * shots, 'total_shots': 2 * shots * repeats,
                     'energies': values, 'exact_energy': exact,
                     'mean': float(np.mean(values)), 'sample_std': float(np.std(values, ddof=1)),
                     'min': min(values), 'max': max(values)})
    return rows


def optimize_vqe(objective, initial=(0.4, 0.6)):
    initial = np.array([_real(x, 'Initial angle') for x in initial], dtype=float)
    if initial.shape != (2,):
        raise ValueError('Initial values must contain theta and phi.')
    history = []
    def observed(parameters):
        point = tuple(map(float, parameters))
        energy = _real(objective(np.asarray(parameters).copy()), 'Objective return value')
        history.append({'parameters': point, 'energy': energy})
        return energy
    result = minimize(observed, initial, method='BFGS', options={'gtol': 1e-6, 'maxiter': 120})
    return {'parameters': tuple(map(float, result.x)), 'objective_energy': float(result.fun),
            'nfev': int(result.nfev), 'nit': int(result.nit), 'success': bool(result.success),
            'message': str(result.message), 'initial': tuple(map(float, initial)), 'history': history}


def _run_check(label, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[label] = 'Incomplete'
        print(f'{label}: Incomplete — complete the TODO and run again.')
        return False
    except Exception as error:
        CHECK_STATUS[label] = 'Failed'
        print(f'{label}: Failed — {type(error).__name__}: {error}')
        return False
    CHECK_STATUS[label] = 'Passed'
    print(f'{label}: Passed')
    return True


def check_energy_from_expectations(function):
    def check():
        for vector in ([1, 0, 0, 0], [1, 2j, -0.7, 0.3j], [0.4j, -0.8, 1.1j, 0.6]):
            vector = np.asarray(vector, dtype=complex)
            state = Statevector(vector / np.linalg.norm(vector))
            values = _expectations_from_state(state)
            for coefficient in (-1.0, 1.0, 0.37):
                actual = _real(function(dict(values), xy_coeff=coefficient))
                expected = float(np.real(state.expectation_value(hamiltonian(coefficient))))
                assert np.isclose(actual, expected, atol=ATOL, rtol=0), 'Check the coefficients, signs, and dictionary keys.'
        estimated = {'ZZ': 0.9, 'YX': 0.8, 'IZ': 0.9, 'ZI': 0.9}
        for coefficient, expected in ((-1.0, -2.6), (1.0, -1.0), (0.5, -1.4)):
            actual = _real(function(dict(estimated), xy_coeff=coefficient))
            assert np.isclose(actual, expected, atol=ATOL, rtol=0), 'Use the supplied values and coefficients for sample estimates as well.'
    return _run_check(CHECK_LABELS[0], check)


def check_phase_ansatz(function):
    def check():
        for theta, phi in ((0.0, 0.8), (0.43, -0.79), (1.31, 2.17), (-0.63, 0.92)):
            state = circuit_state(function(theta, phi), require_ansatz=True)
            target = Statevector([np.cos(theta / 2), 0, 0, np.exp(1j * phi) * np.sin(theta / 2)])
            assert state_fidelity(state, target) >= 1 - ATOL, 'Check the target amplitudes and relative phase. A global phase difference is allowed.'
    return _run_check(CHECK_LABELS[1], check)


def _check_objective_wiring(function):
    "Keep the answer intact and check that called tools' return values are used in sequence."
    if not isinstance(function, FunctionType):
        raise AssertionError('Write the objective inside the provided function definition.')
    point = (0.37, -0.58)
    probe = QuantumCircuit(2)
    probe.ry(0.271, 0)
    probe.ry(-0.38, 1)
    probe.cx(1, 0)
    values = {'ZZ': 0.12, 'YX': -0.27, 'IZ': 0.08, 'ZI': 0.31}
    marker = -0.76329
    calls = {'phase': 0, 'expectations': 0, 'energy': 0}
    def phase(theta, phi):
        assert np.allclose([theta, phi], point, atol=ATOL, rtol=0), 'Check the input order theta, phi.'
        calls['phase'] += 1
        return probe
    def expectations(circuit):
        assert state_fidelity(circuit_state(circuit), circuit_state(probe)) >= 1 - ATOL, 'Use the circuit returned by your phase_ansatz.'
        calls['expectations'] += 1
        return dict(values)
    def energy(received, xy_coeff=-1.0):
        assert set(received) == set(values) and all(np.isclose(received[k], values[k], atol=ATOL, rtol=0) for k in values), "Pass the provided tool's expectation values to the energy evaluator."
        assert xy_coeff == -1.0, "Use the base Hamiltonian's XY coefficient."
        calls['energy'] += 1
        return marker
    replacements = {'phase_ansatz': phase, 'expectations_from_circuit': expectations,
                    'energy_from_expectations': energy}
    with _instrument_dependencies(function, replacements):
        actual = _real(function(np.array(point)))
        assert all(calls.values()), 'Connect the circuit, expectation-value tool, and energy evaluator.'
        assert np.isclose(actual, marker, atol=ATOL, rtol=0), 'Return the value produced by your energy evaluator from the objective.'
    return calls



def check_vqe_objective(function):
    def check():
        for point in ((0.4, 0.6), (1.2, -0.7), (-0.8, 1.4)):
            actual = _real(function(np.array(point)))
            state = Statevector([np.cos(point[0] / 2), 0, 0, np.exp(1j * point[1]) * np.sin(point[0] / 2)])
            expected = float(np.real(state.expectation_value(hamiltonian())))
            assert np.isclose(actual, expected, atol=ATOL, rtol=0), 'Check the connection between the circuit and energy at the input angles.'
        _check_objective_wiring(function)
    return _run_check(CHECK_LABELS[2], check)


def run_demo(label, experiment, requires=()):
    LAB_RESULTS.pop(label, None)
    missing = [key for key in requires if CHECK_STATUS.get(key) != 'Passed']
    if missing:
        print(f'[Waiting] {label}: first pass these code checks: {", ".join(missing)}')
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f'[Waiting] {label}: complete the earlier TODOs.')
        return None
    except Exception as error:
        print(f'[Experiment error] {label}: {type(error).__name__}: {error}')
        return None
    LAB_RESULTS[label] = result
    return result

In [ ]:
RUN_OBSERVATIONS = False
print('Run observation experiments:', RUN_OBSERVATIONS)
print('In the student notebook, write predictions and calculations first, then set this to True and run the connected experiment cells.')

## Problem 1. How Do Measurements Become an Energy?

In this problem, you will build <strong>a way to evaluate a circuit's prepared state as a single energy value</strong>. Later, the classical optimizer will compare these energies while trying different angles. First, you need to identify the observables and the signs and coefficients used to combine their results.

The Hamiltonian for this lab is

$$
\mathcal H
=
-Z_0Z_1-X_0Y_1-\frac12(Z_0+Z_1)
$$

Once you have the expectation values of its four terms, you can combine them with the same coefficients to calculate the state's energy.

---

### Reading the Terms of the Hamiltonian

First, distinguish the meaning of subscripts from the meaning of products.

$Z_0$ is the Z observable on q0. As an operator on the full two-qubit system, it applies Z to q0 and the identity to q1. Conversely, $Z_1$ is the Z observable on q1.

$Z_0Z_1$ is a joint observable formed by multiplying the two qubits' Z measurement values. Measure both qubits in Z in the same trial, interpret each result as $+1$ or $-1$, and multiply them. Equal values give $+1$; different values give $-1$.

Read $X_0Y_1$ in the same way: measure q0 in X and q1 in Y, then <strong>multiply the two values from the same trial</strong>.

The expectation-value notation $\langle P\rangle$ means the average result of measuring observable $P$ in the current state. For example,

$$
\langle Z_0Z_1\rangle
=
\langle\psi|Z_0Z_1|\psi\rangle
$$



Distinguish <strong>averaging the product of two measurement values</strong> from <strong>averaging each value separately and then multiplying the two averages</strong>.

$$
\langle Z_0Z_1\rangle
\quad\text{and}\quad
\langle Z_0\rangle\langle Z_1\rangle
$$

These are generally different. A joint expectation value includes correlations between the two measurement results. To calculate the $Z_0Z_1$ term, therefore, multiply the results from each matching trial before averaging.

Taking the expectation value of each Hamiltonian term gives the energy

$$
E
=
-\langle Z_0Z_1\rangle
-\langle X_0Y_1\rangle
-\frac12\langle Z_0\rangle
-\frac12\langle Z_1\rangle
$$



Even if the expectation values come from separate experiments, they must all describe <strong>the same prepared state</strong>. Combining values from states prepared with different angles does not calculate the energy of a single state.

---

### Worked Example: The Energy of $\lvert00\rangle$

Before handling all four terms in general, check their signs and coefficients for a simple state.

Measuring both qubits of $|00\rangle$ in Z always gives bit 0. Since bit 0 corresponds to the Z value $+1$,

$$
\langle Z_0\rangle=1,\qquad
\langle Z_1\rangle=1,\qquad
\langle Z_0Z_1\rangle=1
$$



Measuring $|0\rangle$ in X or Y gives $+1$ and $-1$ with equal probability. Because $|00\rangle$ is a product state, the X result on q0 and the Y result on q1 are also independent. Their product therefore gives $+1$ and $-1$ with equal probability, so

$$
\langle X_0Y_1\rangle=0
$$



Substituting these values into the energy expression gives

$$
E_{|00\rangle}
=
-1-0-\frac12-\frac12
=
-2
$$



This value is the first reference for the energy function you will write. The same rule must work for expectation values from states other than $|00\rangle$ as well.

---

### Reading Measurement Bits as Observable Values

Qiskit records measurement bits $0,1$, rather than the values $+1,-1$. To calculate Pauli expectation values, interpret the bits as follows.

$$
b=0\;\longleftrightarrow\;+1,
\qquad
b=1\;\longleftrightarrow\;-1.
$$

In a single expression, the observable value is $(-1)^b$.

If the results on q0 and q1 are $b_0,b_1$, the values from one trial in the Z setting are

$$
z_0=(-1)^{b_0},\qquad
z_1=(-1)^{b_1},\qquad
z_0z_1=(-1)^{b_0+b_1}
$$



After $N$ trials in the same setting, average the value from each trial to estimate the expectation value. For example,

$$
\widehat{\langle Z_0Z_1\rangle}
=
\frac1N\sum_{j=1}^{N}z_0^{(j)}z_1^{(j)}
$$

The hat $\widehat{\phantom{E}}$ indicates an <strong>estimate</strong> obtained from a finite measurement record.

In this expression, $z_0^{(j)}$ and $z_1^{(j)}$ come from the same trial $j$. Do not shuffle each qubit's results separately or arbitrarily pair results from different trials.

The first part of this lab uses the provided tools to calculate exact expectation values. Later, with finite shots, you will use the same principle of interpreting measurement bits as observable values and averaging them.

---

### Using Z Measurements to Read X and Y Observables

Even when the circuit ends with a Z measurement, a change of basis beforehand lets you read the original state's X or Y observable.

Start with X measurement. The two X eigenstates are

$$
|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt2},
\qquad
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}
$$

They correspond to values $+1,-1$, respectively. The Hadamard gate acts as

$$
H|+\rangle=|0\rangle,
\qquad
H|-\rangle=|1\rangle
$$

Thus, <strong>applying H and then measuring Z</strong> lets you read an X measurement of the original state as bits 0 and 1.

Y measurement uses the same idea. The two Y eigenstates are

$$
|+i\rangle=\frac{|0\rangle+i|1\rangle}{\sqrt2},
\qquad
|-i\rangle=\frac{|0\rangle-i|1\rangle}{\sqrt2}
$$

They correspond to Y values $+1,-1$, respectively.

The inverse phase gate $S^\dagger$ acts as

$$
S^\dagger|0\rangle=|0\rangle,
\qquad
S^\dagger|1\rangle=-i|1\rangle
$$

This gives

$$
|+i\rangle
\xrightarrow{S^\dagger}
|+\rangle
\xrightarrow{H}
|0\rangle,
$$

$$
|-i\rangle
\xrightarrow{S^\dagger}
|-\rangle
\xrightarrow{H}
|1\rangle
$$

To measure Y, therefore, <strong>apply $S^\dagger$ first, then H, and finally measure Z</strong>.

Here, the basis change prepares the original state's observable to be read. The bit obtained after H on q0 represents the original X value; the bit obtained after $S^\dagger$ and H on q1 represents the original Y value.

This lab uses the following two measurement settings.

| Measurement setting | Operation before measurement on q0 | Operation before measurement on q1 | Expectation values obtained from the record |
|---|---|---|---|
| <strong>Z setting</strong> | None | None | $\langle Z_0\rangle,\ \langle Z_1\rangle,\ \langle Z_0Z_1\rangle$ |
| <strong>XY setting</strong> | H | $S^\dagger$ followed by H | $\langle X_0Y_1\rangle$ |

In the Z setting, the two results from one trial give $z_0,z_1,z_0z_1$. You can therefore calculate all three expectation values from the same measurement record.

In the XY setting, multiply the X and Y values from each trial, then average them. For this setting, <strong>prepare a fresh copy of the same state used for the Z setting</strong>. Applying the basis change after measuring the data in Z would probe the post-measurement state, not the original state.

---

### Matching Qubit Subscripts to Qiskit Strings

In equations, subscripts identify the qubits on which operators act. In a Qiskit Pauli string, each character's position identifies the qubit.

For two-qubit Pauli strings, <strong>q1 is on the left and q0 is on the right</strong>. The mapping is therefore:

| Mathematical notation | Qiskit Pauli string | Operation on each qubit |
|---|---|---|
| $Z_0Z_1$ | `"ZZ"` | Z on q1, Z on q0 |
| $X_0Y_1$ | `"YX"` | Y on q1, X on q0 |
| $Z_0$ | `"IZ"` | Identity on q1, Z on q0 |
| $Z_1$ | `"ZI"` | Z on q1, identity on q0 |

In particular, do not write `"XY"` just because the equation says $X_0Y_1$. `"XY"` is a different observable: X on q1 and Y on q0.

The statevector basis also uses

$$
|q_1q_0\rangle
$$

order. For example, $|01\rangle$ means q1 is in $|0\rangle$ and q0 is in $|1\rangle$.

Use this ordering consistently in the exercises below.

---

### Your Turn

### (a) Calculating the Energies of Different States

Consider the following two states.

$$
|\psi_A\rangle
=
|0\rangle_{q_1}|1\rangle_{q_0}
=
|01\rangle,
$$

$$
|\psi_B\rangle
=
|+i\rangle_{q_1}|+\rangle_{q_0}.
$$

Calculate the four expectation values and the energy of each state, then complete the table.

| State | $\langle Z_0Z_1\rangle$ | $\langle X_0Y_1\rangle$ | $\langle Z_0\rangle$ | $\langle Z_1\rangle$ | Energy $E$ |
|---|---:|---:|---:|---:|---:|
| $\lvert\psi_A\rangle$ | … | … | … | … | … |
| $\lvert\psi_B\rangle$ | … | … | … | … | … |

After completing the table, explain the following.

- If you interchange the states of q0 and q1 when reading $|\psi_A\rangle$, which expectation values in the table change? Would comparing only the final energy always reveal this mistake?
- If you use the Qiskit string `"XY"` to calculate the $X_0Y_1$ expectation value of $|\psi_B\rangle$, which observable do you actually calculate? Check whether it gives the same result as the intended observable for this state.

Compare your calculations with the provided tool. Check not only the energy, but also <strong>whether each expectation value refers to the intended qubits and observable</strong>.

---

### Answer 1a

Prediction and calculations before running:

…

Observations and explanation:

…

### Observation Experiment 1a

Write your hand calculations before running this cell. The experiment compares them with the provided physics tools. If its output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_1a():
    first = QuantumCircuit(2)
    first.x(0)
    second = QuantumCircuit(2)
    second.h(0)
    second.h(1)
    second.s(1)
    results, rows = [], []
    for label, circuit in (("A: |01>", first), ("B: |+i> on q1, |+> on q0", second)):
        values = expectations_from_circuit(circuit)
        energy = exact_energy(circuit)
        state = circuit_state(circuit)
        other_xy = float(np.real(state.expectation_value(SparsePauliOp("XY"))))
        results.append({"label": label, "values": values, "energy": energy, "wrong_xy": other_xy})
        rows.append([label, *[values[k] for k in OBSERVABLE_KEYS], energy, other_xy])
    show_table(['State', "ZZ", "YX", "IZ", "ZI", 'Exact E', 'Comparison: XY'], rows)
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("1a", experiment_1a)
else:
    LAB_RESULTS.pop("1a", None)
    print('[Waiting] 1(a): write your prediction and calculations, then set RUN_OBSERVATIONS to True.')

### (b) Writing a Function That Converts Expectation Values to Energy

Write the following function.

```python
energy_from_expectations(values, xy_coeff=-1.0)
```

`values` is a dictionary containing the expectation values of four observables.

| Key | Stored value |
|---|---|
| `"ZZ"` | $\langle Z_0Z_1\rangle$ |
| `"YX"` | $\langle X_0Y_1\rangle$ |
| `"IZ"` | $\langle Z_0\rangle$ |
| `"ZI"` | $\langle Z_1\rangle$ |

`xy_coeff` is the coefficient of the $X_0Y_1$ term. Its default value $-1.0$ corresponds to the current Hamiltonian. If another value $c$ is supplied, calculate the energy of

$$
\mathcal H(c)
=
-Z_0Z_1+cX_0Y_1-\frac12(Z_0+Z_1)
$$

The coefficients of the other three terms stay unchanged.

The function must <strong>return one real number: the energy calculated from the supplied expectation values and coefficient</strong>. Do not prepare states, measure, or optimize inside this function.

First check the signs and coefficients using the earlier $|00\rangle$ example. Then check the two states in (a) and the additional provided states. Use the four supplied values rather than directly returning a known energy for a particular state.

The check cell compares the return value with an independent Hamiltonian expectation calculation for several states and coefficients. Once complete, this same evaluator will be used in later circuit experiments and optimization.

---

In [ ]:
def energy_from_expectations(values, xy_coeff=-1.0):
    # TODO BEGIN — Calculate energy from the four supplied expectation values and coefficient.
    raise NotImplementedError('energy_from_expectations')
    # TODO END

In [ ]:
_ = check_energy_from_expectations(energy_from_expectations)

### Answer 1b

Calculations, observations, and reasoning:

…

### (c) Designing the Measurements Needed to Evaluate Energy

You want to measure the energy of a state prepared at one set of angles. Run the Z setting $N$ times and the XY setting $N$ times, preparing the state afresh and measuring both qubits in every trial.

Complete the following table.

| Expectation value | Measurement setting | Value used from one trial | Number of trials in the average |
|---|---|---|---:|
| $\langle Z_0\rangle$ | … | … | … |
| $\langle Z_1\rangle$ | … | … | … |
| $\langle Z_0Z_1\rangle$ | … | … | … |
| $\langle X_0Y_1\rangle$ | … | … | … |

Then examine the following three claims. Decide whether each is appropriate; if not, rewrite it with the correct calculation or procedure.

1. <strong>“There are four expectation values, so evaluating the energy once requires a total of $4N$ state preparations.”</strong>  
   Explain which expectation values can be obtained from the same measurement record.

2. <strong>“To reduce the number of state preparations, I will apply H and $S^\dagger$ to the two qubits after their Z measurements, then measure the XY setting.”</strong>  
   Explain which state's observables the second setting would read. How should the experiment be organized to obtain the energy of the originally prepared state?

3. <strong>“The expectation value of $Z_0Z_1$ can be obtained by multiplying q0's average Z value by q1's average Z value.”</strong>  
   Compare this with multiplying the two results from each trial before averaging. Explain whether the methods agree in general, using a state from an earlier lab as an example.

Finally, in one or two sentences, state <strong>what you should record to check that the four values supplied to the energy evaluator describe the same state</strong>.

### Answer 1c

Calculations, observations, and reasoning:

…

## Problem 2. Can Changing the Angle Reach the Ground State?

In the previous problem, you built an evaluator that combines four expectation values into one energy. Now use it to investigate <strong>how the energy changes when the circuit angle changes</strong>.

Can trying enough angles prepare any state? The circuit structure determines which states its angles can produce. Before searching for a low energy, you therefore need to identify the states this circuit can prepare.

Start with the circuit from the review material. Prepare two qubits in $|00\rangle$ and apply gates in the following order.

1. Apply $R_y(\theta)$ to q0.
2. Apply CX with q0 as the control and q1 as the target.

This circuit has one adjustable parameter, the angle $\theta$. The circuit code is provided; you will calculate <strong>the state it produces and that state's energy</strong>. Use the result to decide whether changing the angle alone can reach the ground energy.

---

### Rotation Gates Change Superposition Coefficients

First examine the state produced by $R_y(\theta)$. Its matrix is

$$
R_y(\theta)
=
\begin{pmatrix}
\cos(\theta/2)&-\sin(\theta/2)\\
\sin(\theta/2)&\cos(\theta/2)
\end{pmatrix}
$$

Applying it to $|0\rangle$ gives

$$
R_y(\theta)|0\rangle
=
\cos\frac{\theta}{2}|0\rangle
+
\sin\frac{\theta}{2}|1\rangle
$$



Changing $\theta$ changes the coefficients of the two basis components. Their squares are the probabilities of measuring 0 and 1 in Z, and

$$
\cos^2\frac{\theta}{2}
+
\sin^2\frac{\theta}{2}
=1
$$

so the state is normalized.

In this circuit, only q0 receives the rotation. Immediately afterward, q1 is still in $|0\rangle$. When writing the joint state, <strong>include both q1's state and the rotated state of q0</strong>.

As in the previous problem, the statevector basis uses $|q_1q_0\rangle$ order. A change in q0's component changes the rightmost bit in the ket.

---

### CX Acts on Each Component of a Superposition

Next apply CX. Here, q0 is the control and q1 is the target, so q1 is flipped in any component where q0 is 1.

In $|q_1q_0\rangle$ order, its action on basis states is:

| Before CX | After CX |
|---|---|
| $\lvert00\rangle$ | $\lvert00\rangle$ |
| $\lvert01\rangle$ | $\lvert11\rangle$ |
| $\lvert10\rangle$ | $\lvert10\rangle$ |
| $\lvert11\rangle$ | $\lvert01\rangle$ |

For a superposition, apply this rule to each component and retain its original coefficient. For two distinct basis states $|u\rangle,|v\rangle$, for example,

$$
\mathrm{CX}
\bigl(a|u\rangle+b|v\rangle\bigr)
=
a\,\mathrm{CX}|u\rangle
+
b\,\mathrm{CX}|v\rangle
$$



In this circuit, $R_y$ adjusts the coefficients of q0's two components. CX then changes q1 according to q0's component, creating a relationship between their measurement results. You will apply the rules above to derive the resulting joint state.

Including CX does not produce entanglement at every angle. Being able to create some entangled states also does not mean being able to create every entangled state. Here, you will identify <strong>the set of states this particular circuit and its parameter can prepare</strong>.

---

### Distinguishing Coefficient Magnitudes from Relative Phase

When examining a circuit's state, check the signs and phases of the coefficients as well as the probabilities of the components.

Consider a superposition of two orthogonal basis states,

$$
|\chi\rangle=a|u\rangle+b|v\rangle
$$

Here, $|a|^2,|b|^2$ are the probabilities of the two components in this basis. These probabilities do not fully determine $a,b$. For example, multiplying $b$ by a phase $e^{i\phi}$ leaves $|b|^2$ unchanged.

$$
a|u\rangle+b|v\rangle,
\qquad
a|u\rangle+e^{i\phi}b|v\rangle
$$

These states can give the same measurement probabilities in this basis while having different relative phases between their components.

Every matrix entry of $R_y$ is real, and CX permutes basis states without adding complex phases. Starting from $|00\rangle$, this circuit therefore has real computational-basis amplitudes.

Real coefficients can be positive or negative, so changing the angle can change the sign between components. Distinguish this from being able to adjust an arbitrary complex relative phase.

You will calculate the Hamiltonian terms to see <strong>whether this restriction on the coefficients also affects the energy</strong>. In particular, $X_0Y_1$ includes a Y observable, so retain the complex phases that appear in the calculation.

---

### Calculating an Expectation Value from a State

To find an expectation value, first apply the observable to the state, then take the inner product of the result with the original state.

$$
\langle P\rangle
=
\langle\psi|\bigl(P|\psi\rangle\bigr).
$$

As an example, consider the normalized one-qubit state

$$
|\chi\rangle=a|0\rangle+b|1\rangle
$$

Applying Z gives

$$
Z|\chi\rangle=a|0\rangle-b|1\rangle
$$

and the bra of the original state is

$$
\langle\chi|
=
a^*\langle0|+b^*\langle1|
$$

Using the fact that distinct basis states have zero inner product gives

$$
\langle Z\rangle
=
(a^*\langle0|+b^*\langle1|)
(a|0\rangle-b|1\rangle)
=
|a|^2-|b|^2
$$



The same method works for two qubits. The required Pauli actions are:

| Operator | Applied to $\lvert0\rangle$ | Applied to $\lvert1\rangle$ |
|---|---|---|
| $X$ | $\lvert1\rangle$ | $\lvert0\rangle$ |
| $Y$ | $i\lvert1\rangle$ | $-i\lvert0\rangle$ |
| $Z$ | $\lvert0\rangle$ | $-\lvert1\rangle$ |

For example, to apply $X_0Y_1$, apply X to q0 and Y to q1. Multiply each component's coefficient by any resulting phase, then take the inner product with the original state.

When Y is involved, $i$ may appear in intermediate steps. Do not discard it or replace coefficients with their absolute values. The final expectation value of a Pauli observable is real, but its calculation uses complex coefficients and their conjugates together.

---

### Distinguishing an Angle Scan from the Circuit's Overall Limit

Write the prepared state as $|\psi_{\mathrm r}(\theta)\rangle$ and its energy as

$$
E_{\mathrm r}(\theta)
=
\langle\psi_{\mathrm r}(\theta)|
\mathcal H
|\psi_{\mathrm r}(\theta)\rangle
$$

The subscript r identifies this real-amplitude circuit.

The provided scan runs the circuit at several angles and displays:

- Exact expectation values of the four Pauli observables
- Energy calculated by the function you wrote in the previous problem
- The difference from the reference ground energy $E_0$

This experiment has no finite-shot sampling fluctuations. Changes in the table therefore reflect changes in the prepared state and its expectation values as the angle varies.

Testing finitely many angles does not prove the minimum over all angles. A lower value could lie between the sampled points.

This circuit is simple enough to derive its energy explicitly as a function of $\theta$. <strong>Use the scan to check the equation's predictions, and use the equation to determine the minimum over the full angle range.</strong>

The following identities may help simplify your calculation.

$$
\cos^2\frac{\theta}{2}
-
\sin^2\frac{\theta}{2}
=
\cos\theta,
$$

$$
2\sin\frac{\theta}{2}\cos\frac{\theta}{2}
=
\sin\theta.
$$

You can then use the ranges of $\sin\theta,\cos\theta$ to determine how low the energy can go.

---

### Your Turn

### (a) Deriving the State Prepared by the Circuit

The initial state is

$$
|00\rangle=|0\rangle_{q_1}|0\rangle_{q_0}
$$

Derive the state after applying $R_y(\theta)$ to q0 and then $\mathrm{CX}(0,1)$, one step at a time.

| Stage | Joint state of the two qubits |
|---|---|
| Initial state | $\lvert00\rangle$ |
| After $R_y(\theta)$ on q0 | … |
| After $\mathrm{CX}(0,1)$ | … |

Call the state in the last row $|\psi_{\mathrm r}(\theta)\rangle$.

After deriving the state, explain the following.

- Which components among $|00\rangle,|01\rangle,|10\rangle,|11\rangle$ can appear in the final state?
- How do their coefficients change as $\theta$ changes?
- Can this circuit assign an arbitrary complex relative phase between its two nonzero components? Explain using the actions of its gates.

Use the provided state-calculation tool to check a few angles, but write <strong>the state expression for general $\theta$</strong>, rather than only a vector at one angle.

---

### Provided Circuit and State-Inspection Tool

`real_ansatz(theta)` returns the RY → CX circuit for this problem. Use `circuit_state(real_ansatz(theta)).data` to inspect its statevector. Components are ordered `00, 01, 10, 11`. Write the general state expression first, then check it at specific angles.

### Answer 2a

Calculations, observations, and reasoning:

…

### (b) Deriving the Energy as a Function of the Angle

Using the state from (a), express the following four expectation values as functions of $\theta$.

| Observable | Expectation value in $\lvert\psi_{\mathrm r}(\theta)\rangle$ |
|---|---|
| $Z_0Z_1$ | … |
| $X_0Y_1$ | … |
| $Z_0$ | … |
| $Z_1$ | … |

Show your calculation of the $X_0Y_1$ term in this order:

1. Apply $X_0Y_1$ to each basis component of the final state.
2. Add the results to obtain $X_0Y_1|\psi_{\mathrm r}(\theta)\rangle$.
3. Take the inner product with the bra of the original state.

Next, combine the four expectation values with the Hamiltonian's coefficients and simplify

$$
E_{\mathrm r}(\theta)
=
-\langle Z_0Z_1\rangle
-\langle X_0Y_1\rangle
-\frac12\langle Z_0\rangle
-\frac12\langle Z_1\rangle
$$

to a single expression.

Identify <strong>which expectation values change with the angle and which remain constant</strong>. In particular, use your calculation to explain whether the $X_0Y_1$ result holds only at selected angles or at every angle in this state family.

Finally, run the provided scan and check that its four expectation values and energies agree with your equations.

---

### Answer 2b

Prediction and calculations before running:

…

Observations and explanation:

…

### Observation Experiment 2bc

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. Use this table both to check the expression in 2(b) and to assess the minimum in 2(c). If the result says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_2bc():
    rows, results = [], []
    for theta in np.linspace(0, 2*np.pi, 13):
        circuit = real_ansatz(theta)
        values = expectations_from_circuit(circuit)
        energy = energy_from_expectations(values)
        results.append({"theta": float(theta), "values": values, "energy": float(energy)})
        rows.append([theta, *[values[k] for k in OBSERVABLE_KEYS], energy])
    show_table(["theta", "ZZ", "YX", "IZ", "ZI", "Student function's E"], rows)
    best = min(results, key=lambda row: row["energy"])
    show_table(['Theta selected from scan', 'Lowest scanned E', 'Exact E0'],
               [[best["theta"], best["energy"], ground_reference()["energy"]]])
    print('This table is a finite angle scan. Use your equation to determine the minimum over all angles.')
    return {"rows": results, "best": best}

if RUN_OBSERVATIONS:
    _ = run_demo("2bc", experiment_2bc, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("2bc", None)
    print('[Waiting] 2(b,c): write the state and energy expressions before running the observation experiment.')

In [ ]:
if RUN_OBSERVATIONS and "2bc" in LAB_RESULTS:
    reference_energy = ground_reference()["energy"]
    show_table(["theta", "E-E0"],
               [[row["theta"], row["energy"] - reference_energy]
                for row in LAB_RESULTS["2bc"]["rows"]])
else:
    print('[Waiting] The energy-gap table appears after the angle scan has run.')

### (c) Determining the Lowest Energy This Circuit Can Reach

Use the energy expression from (b) to find

$$
E_{\mathrm r,\min}
=
\min_\theta E_{\mathrm r}(\theta).
$$

State the minimum, explain why it is the minimum, and give one angle that achieves it. Do more than copy the smallest value in the scan: use the range of the trigonometric function in your energy expression.

Then complete the following table.

| Comparison | Result and reasoning |
|---|---|
| Lowest energy observed in the scan | … |
| $E_{\mathrm r,\min}$ derived from the equation | … |
| Reference ground energy $E_0$ | $-1-\sqrt2$ |
| Difference between the minima, $E_{\mathrm r,\min}-E_0$ | … |
| Can the current circuit reach the ground energy? | … |

Use these results to evaluate the following two claims.

> <strong>Claim A:</strong> “We simply have not found a good angle yet. Trying more angles with the same circuit will reach the ground energy.”

> <strong>Claim B:</strong> “This circuit contains CX and can create entangled states, so it is expressive enough to prepare the ground state.”

Decide whether each claim is valid here and explain using <strong>the circuit's state and energy expressions</strong>. Do not rely only on the performance or runtime of a particular optimizer.

If you conclude that the circuit needs to change, identify <strong>which property of the state cannot be adjusted with the current angle $\theta$</strong>. In the next problem, you will extend the circuit to control that property.

### Answer 2c

Calculations, observations, and reasoning:

…

## Problem 3. How Can We Change the Circuit to Prepare the Needed State?

In the previous problem, you examined the states prepared by the first circuit as $\theta$ changes. You then derived its energy expression and compared its minimum with the ground energy $E_0$.

Now you will <strong>introduce a new parameter to control the relative phase that the first circuit could not adjust</strong>. Connect the revised circuit to your energy evaluator and use the classical optimizer to search over both angles.

Adding a gate is not the end of the task. You will also check that the new angle actually changes the intended relative phase and that the optimizer uses your circuit and energy function.

---

### Controlling Magnitudes and Relative Phase Separately

The previous circuit prepared

$$
|\psi_{\mathrm r}(\theta)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
\sin\frac{\theta}{2}|11\rangle
$$

Changing $\theta$ changes the two coefficients, but the computational-basis amplitudes remain real.

Now build a circuit that prepares the following state.

$$
|\psi(\theta,\phi)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
e^{i\phi}\sin\frac{\theta}{2}|11\rangle.
$$

The new angle $\phi$ controls the phase attached to the $|11\rangle$ component. Setting $\phi=0$ recovers the previous state expression.

The two parameters have the following roles.

| Parameter | Property it controls |
|---|---|
| $\theta$ | Magnitudes of the $\lvert00\rangle$ and $\lvert11\rangle$ coefficients |
| $\phi$ | Additional relative phase between the two components |

A Z measurement of this state gives

$$
P(00)=\cos^2\frac{\theta}{2},
\qquad
P(11)=\sin^2\frac{\theta}{2}
$$

and zero probability for the other two outcomes. Since $|e^{i\phi}|=1$, these probabilities do not depend on $\phi$.

Thus, with $\theta$ fixed, changing only $\phi$ can <strong>change the joint state without changing the Z measurement probabilities</strong>. You will examine the Hamiltonian's $X_0Y_1$ expectation value and the energy to see this change.

If either component has zero coefficient, the relative phase between the two components cannot be observed in that state. To investigate $\phi$, use a value of $\theta$ for which both components are present.

---

### $R_z$ Assigns Different Phases to the Two Basis Components

Use $R_z(\phi)$ to control the relative phase. This gate is

$$
R_z(\phi)
=
\begin{pmatrix}
e^{-i\phi/2}&0\\
0&e^{i\phi/2}
\end{pmatrix}
$$

and acts on basis states as follows.

$$
R_z(\phi)|0\rangle=e^{-i\phi/2}|0\rangle,
\qquad
R_z(\phi)|1\rangle=e^{i\phi/2}|1\rangle.
$$

Applied to a one-qubit superposition, it gives

$$
R_z(\phi)
\bigl(a|0\rangle+b|1\rangle\bigr)
=
ae^{-i\phi/2}|0\rangle
+
be^{i\phi/2}|1\rangle
$$

Factoring the common term $e^{-i\phi/2}$ out of both components gives

$$
=
e^{-i\phi/2}
\bigl(a|0\rangle+e^{i\phi}b|1\rangle\bigr)
$$



Two different types of phase appear in this expression.

- The factor $e^{-i\phi/2}$ outside the parentheses is a <strong>global phase</strong> multiplying the whole state.
- The factor $e^{i\phi}$ inside the parentheses changes the <strong>relative phase</strong> between components.

Multiplying the entire state by one common phase does not change measurement probabilities. A relative phase, however, can affect measurements in other bases.

A circuit that differs from the target state only by a global phase therefore counts as preparing the same state. A different relative phase does not, even when the Z probabilities agree.

---

### The Position of a Gate Matters

Knowing that $R_z$ controls phase does not mean that inserting it anywhere in the circuit has the same effect.

Follow the circuit and check two things:

1. Which basis components are present when $R_z$ is applied?
2. Do they receive different phases, or does one common phase multiply the entire state?

In this circuit, $R_y$ creates a superposition and CX connects the components of the two qubits. The final effect of $R_z$ can depend on both where it is inserted and which qubit it acts on.

When designing the circuit, do more than list gate names: check <strong>how the state changes at each step</strong>. Different gate placements may prepare the same final state, so checks are based on the prepared state rather than one specific layout.

---

### Connecting the Circuit and Energy Evaluator to an Objective

Once the circuit can prepare states from two angles, you can build the objective function passed to the classical optimizer.

At each evaluation, the optimizer proposes a parameter pair

$$
\boldsymbol\theta=(\theta,\phi).
$$

The objective receives these values and performs the following steps.

| Step | Action |
|---|---|
| 1 | Read $\theta,\phi$ from the input. |
| 2 | Build the state-preparation circuit for those angles. |
| 3 | Calculate the state's four Pauli expectation values. |
| 4 | Combine them with the energy evaluator from Problem 1. |
| 5 | Return the calculated energy. |

The provided tool for this problem is

```python
expectations_from_circuit(circuit)
```

It calculates the state prepared by the input circuit from $|00\rangle$ and returns a dictionary with four keys:

```python
"ZZ", "YX", "IZ", "ZI"
```

The keys have the same meanings as in Problem 1. The returned values are <strong>exact expectation values</strong> of an ideal state, without finite-shot sampling fluctuations.

Connect this tool between the two functions you wrote.

$$
(\theta,\phi)
\;\longrightarrow\;
\text{state-preparation circuit}
\;\longrightarrow\;
\text{four expectation values}
\;\longrightarrow\;
E.
$$

The optimizer uses the returned energy to choose the next angles to try. You do not need to implement an optimizer yourself.

---

### Optimization History Records the Candidates Evaluated

The provided optimization experiment starts from

$$
(\theta,\phi)=(0.4,0.6)
$$

with angles in radians.

The experiment records the angles passed to the objective and the energy returned at each evaluation. These records show which states the optimizer tried.

The recorded energy does not have to decrease at every evaluation. An optimizer may test nearby candidates to choose its next direction, including candidates with higher energy. It may also call the objective several times in one iteration, so <strong>do not assume the objective-evaluation count equals the iteration count</strong>.

In this problem, record the angles and energy found and check the revised circuit's behavior. The next problem examines the termination message together with whether the ground state was actually reached.

---

### Your Turn

### (a) Writing a Circuit That Prepares the Target State with Two Angles

Write the following function.

```python
phase_ansatz(theta, phi)
```

The function must return a circuit that starts from $|00\rangle$ and prepares

$$
|\psi(\theta,\phi)\rangle
=
\cos\frac{\theta}{2}|00\rangle
+
e^{i\phi}\sin\frac{\theta}{2}|11\rangle.
$$

The function must satisfy the following conditions.

- Inputs `theta` and `phi` are real angles in radians.
- Return a `QuantumCircuit(2)` with two qubits and no classical bits.
- Use $R_y$, $R_z$, and CX gates.
- Do not use `initialize` to specify the state directly, or include measurements.
- Prepare the target state for general supplied $\theta,\phi$, rather than only at selected angles.
- A global phase multiplying the entire target state is allowed.

Start from the previous circuit and add $R_z$ to control the relative phase.

After writing the circuit, explain the following.

1. On which qubit and at which position did you apply $R_z$?
2. What phase does each of the two final components acquire from $R_z$ at that position?
3. After factoring out a common global phase, how does your state match the target expression?

The check cell compares the prepared joint state with the target at several angles. <strong>Matching only Z measurement probabilities is not enough.</strong>

---

In [ ]:
def phase_ansatz(theta, phi):
    circuit = QuantumCircuit(2)
    # TODO BEGIN — Prepare the target joint state for the two angles.
    raise NotImplementedError('phase_ansatz')
    # TODO END
    return circuit

In [ ]:
_ = check_phase_ansatz(phase_ansatz)

### Answer 3a

Calculations, observations, and reasoning:

…

### (b) Connecting Your Circuit to the Classical Optimizer

Write the following function.

```python
vqe_objective(parameters)
```

`parameters` is a length-2 collection in this order:

```python
theta, phi
```

Connect the following tools in your objective.

| Tool | Role |
|---|---|
| Your `phase_ansatz` | Prepare the circuit for the two angles |
| Provided `expectations_from_circuit` | Calculate four expectation values of the circuit's state |
| Your `energy_from_expectations` | Convert the expectation values into one energy |

The coefficient of $X_0Y_1$ in the current Hamiltonian is $-1$. The objective must return this Hamiltonian's energy as one real number.

Do not insert the exact ground energy or known optimal angles into the function. The optimizer calls it at many candidate angles, so <strong>each call must calculate the energy of the state prepared with its input angles</strong>.

After passing the check cell, run the provided optimization experiment and record the following.

| Item | Result |
|---|---|
| Initial values | $(0.4,0.6)$ |
| $\theta,\phi$ returned by the optimizer | … |
| Energy at the returned angles | … |
| Number of objective evaluations | … |
| Optimizer's termination message | … |

Pass the returned angles back to `vqe_objective` and check that the reported energy is reproduced.

Finally, compare this result with the first circuit's minimum energy and describe <strong>how extending the circuit changed the energy</strong>. Along with the numbers, explain which property of the state the new parameter controls.

---

In [ ]:
def vqe_objective(parameters):
    # TODO BEGIN — Connect circuit preparation, expectation calculation, and energy evaluation.
    raise NotImplementedError('vqe_objective')
    # TODO END

In [ ]:
_ = check_vqe_objective(vqe_objective)

### Answer 3b

Calculations, observations, and reasoning:

…

### Observation Experiment 3b

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. If the output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_3b():
    report = optimize_vqe(vqe_objective, initial=(0.4, 0.6))
    recalculated = float(vqe_objective(report["parameters"]))
    assert np.isclose(recalculated, report["objective_energy"], atol=ATOL, rtol=0), 'Recheck the energy at the returned angles.'
    report["recalculated_energy"] = recalculated
    show_table(['Initial values', 'Returned theta, phi', 'Objective E', 'Recalculated E', 'Evaluations', 'Number of repetitions'],
               [[report["initial"], report["parameters"], report["objective_energy"], recalculated, report["nfev"], report["nit"]]])
    print('Optimizer termination:', report["success"], report["message"])
    show_table(['Evaluation order', "theta,phi", 'Energy'],
               [[index+1, row["parameters"], row["energy"]] for index, row in enumerate(report["history"])])
    print('The table follows objective evaluations. Distinguish their count from optimizer iterations.')
    return report

if RUN_OBSERVATIONS:
    _ = run_demo("3b", experiment_3b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("3b", None)
    print('[Waiting] 3(b): check all three functions, then enable observations.')

### (c) Checking Whether the Added Angle Changes Relative Phase

Separately from the optimization result, check whether the new parameter $\phi$ plays its intended role in the circuit.

First fix

$$
\theta=\frac{\pi}{3}
$$

and vary $\phi$ over the following values.

$$
\phi=0,\quad \frac{\pi}{3},\quad \frac{\pi}{2},\quad \pi.
$$

Both components of the target state are present at this $\theta$. Use the provided experiment to inspect your circuit's Z probabilities, $X_0Y_1$ expectation value, and energy.

| $\phi$ | $P(00)$ | $P(11)$ | $\langle X_0Y_1\rangle$ | Energy |
|---|---:|---:|---:|---:|
| $0$ | … | … | … | … |
| $\pi/3$ | … | … | … | … |
| $\pi/2$ | … | … | … | … |
| $\pi$ | … | … | … | … |

Next examine a comparison circuit with this gate order:

1. Starting from $|00\rangle$, apply $R_z(\phi)$ to q0.
2. Apply $R_y(\theta)$ to q0.
3. Apply $\mathrm{CX}(0,1)$.

Run this comparison circuit at the same $\theta$ and four $\phi$ values. It contains $R_z$, but calculate the state changes to decide whether that gate has the same role as in your circuit.

Use the two experiments to explain the following.

- <strong>Your circuit:</strong> How do the Z probabilities and $X_0Y_1$ expectation value respond to changing $\phi$? Use the target state expression to explain why they respond differently.
- <strong>Comparison circuit:</strong> Find the phase attached to $|00\rangle$ by the initial $R_z(\phi)$. After the remaining gates, determine whether it is a global or relative phase in the final state.
- <strong>Verification:</strong> Would Z probabilities alone be sufficient to distinguish the circuits? Explain what the additional observable and state comparisons reveal.

Do not infer different physical states merely because the complex statevector entries look different. A common phase may multiply the whole state. Check <strong>whether the relative phase between components and the actual measurement results changed</strong>.

### Answer 3c

Prediction and calculations before running:

…

Observations and explanation:

…

### Observation Experiment 3c

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. If the output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_3c():
    theta = np.pi / 3
    results, rows = [], []
    for label, function in (('Your circuit', phase_ansatz), ('Initial-RZ comparison circuit', phase_before_rotation)):
        reference = circuit_state(function(theta, 0))
        for phi in (0, np.pi/3, np.pi/2, np.pi):
            circuit = function(theta, phi)
            state = circuit_state(circuit)
            probabilities = state.probabilities()
            values = expectations_from_circuit(circuit)
            energy = energy_from_expectations(values)
            fidelity = float(state_fidelity(reference, state))
            rows.append([label, phi, probabilities[0], probabilities[3], values["YX"], energy, fidelity])
            results.append({"circuit": label, "theta": theta, "phi": phi,
                            "probabilities": probabilities, "values": values,
                            "energy": float(energy), "fidelity_to_phi_zero": fidelity})
    show_table(['Circuit', "phi", "P(00)", "P(11)", "YX", "E", 'F with the same circuit at phi=0'], rows)
    print("The fidelity reference here is each circuit's phi=0 state, not the ground state.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("3c", experiment_3c, (CHECK_LABELS[0], CHECK_LABELS[1]))
else:
    LAB_RESULTS.pop("3c", None)
    print('[Waiting] 3(c): predict the state changes for different RZ positions, then run the observation experiment.')

## Problem 4. What Can We Conclude from an Optimization Result?

You have built a circuit with an adjustable relative phase and connected the energy evaluator to a classical optimizer. Now check <strong>whether the optimizer's result means the ground-state search succeeded</strong>.

Examine three things: why the optimizer stopped, how low the energy is at the returned angles, and how close the prepared state is to the ground state.

These are related but distinct pieces of information. With finite measurements, energy estimates can also differ for the same state. In this problem, interpret <strong>exact state calculations separately from estimates based on measurement records</strong>.

---

### A Termination Message Explains Why the Search Stopped

A classical optimizer stops according to specified numerical conditions. For example, it may judge changes near the current angles to be sufficiently small, or reach a limit on the number of evaluations.

A successful termination message therefore does not mean that all possible angles were searched and the lowest energy found. Conversely, even a message indicating numerical difficulties may accompany angles that already give a good state.

Use the message to understand the execution, and evaluate the result separately. This small two-qubit problem lets you use the exact ground energy and ground state as references.

---

### Checking How Close the Energy Is to the Reference

Let the returned angles be $(\theta_*,\phi_*)$. Prepare the circuit again at those angles and calculate the exact energy

$$
E_*
=
\langle\psi(\theta_*,\phi_*)|
\mathcal H
|\psi(\theta_*,\phi_*)\rangle
$$



Record the difference from the ground energy as

$$
\delta E=E_*-E_0
$$

This measures <strong>how far the energy of the found state lies above the exact ground energy</strong>.

For the exact energy of a normalized state, the variational principle gives

$$
\delta E\ge 0
$$

Numerical calculations can introduce tiny rounding errors, so use the specified tolerance. Do not decide that two values are equal based only on their rounded display.

In this lab, the criterion for comparing exact energies is

$$
|E_*-E_0|\le 10^{-6}
$$

This is a numerical verification criterion for this lab's ideal state preparation and energy calculation.

---

### Checking the Prepared State as Well as Its Energy

This Hamiltonian has a unique ground state. If the exact reference state is $|\psi_0\rangle$ and the circuit prepares $|\psi_*\rangle$, their pure-state fidelity is

$$
F
=
|\langle\psi_0|\psi_*\rangle|^2
$$



For normalized states, $F$ lies between 0 and 1. If $F=1$, the states agree up to a global phase; if $F=0$, they are orthogonal.

For example, if the circuit output is

$$
|\psi_*\rangle=e^{i\chi}|\psi_0\rangle
$$

then

$$
F
=
|e^{i\chi}\langle\psi_0|\psi_0\rangle|^2
=
1
$$

Even if its statevector entries are not literally equal to those of the reference, one common phase on the entire vector represents the same physical state.

Matching computational-basis probabilities, however, does not establish $F=1$. As the previous problem showed, states with different relative phases can have the same Z probabilities.

The provided tool calculates fidelity with the exact ground state. Use the following numerical criterion for state comparison:

$$
F\ge 1-10^{-6}
$$



Considering both the energy difference and fidelity checks whether a low reported energy agrees with the state actually prepared. The reference ground state is used for this verification, not supplied as a target state to the optimizer's objective.

---

### Exact Energy and Measurement-Based Estimates Are Different

So far, the provided tools have calculated exact expectation values. Now fix the optimized angles and estimate the same state's energy from finitely many measurement records.

State preparation and gates remain ideal in this sampling experiment. Only the method for obtaining expectation values changes. Instead of calculating exact probabilities directly, run each setting a fixed number of times and average the observed values.

From the Z-setting record, estimate

$$
\widehat{\langle Z_0Z_1\rangle},
\qquad
\widehat{\langle Z_0\rangle},
\qquad
\widehat{\langle Z_1\rangle}
$$

and from the XY-setting record of a freshly prepared state, estimate

$$
\widehat{\langle X_0Y_1\rangle}
$$

Combining them gives the energy estimate

$$
\widehat E
=
-\widehat{\langle Z_0Z_1\rangle}
-\widehat{\langle X_0Y_1\rangle}
-\frac12\widehat{\langle Z_0\rangle}
-\frac12\widehat{\langle Z_1\rangle}
$$



Even at fixed angles, measurement records can vary, so repeated values of $\widehat E$ can differ slightly. <strong>This does not mean the circuit prepares a better or worse state each time.</strong> The fluctuation comes from estimating the same state's energy with finite records.

The variational bound $E\ge E_0$ applies to the <strong>exact energy expectation value</strong> of a normalized state. The estimate $\widehat E$, formed by combining finite-sample averages from different settings, need not satisfy it on every trial. If one estimate falls below $E_0$, interpret it by distinguishing an exact expectation from a sample estimate.

---

### What Changes When You Increase the Number of Shots?

The provided experiment fixes the angles from the previous problem and repeatedly estimates energy at different shot counts per setting.

| Shots per setting $N$ | Total shots per energy estimate |
|---:|---:|
| 64 | $2N$ |
| 256 | $2N$ |
| 1,024 | $2N$ |

Each energy estimate uses $N$ Z-setting shots and $N$ XY-setting shots. The three Z-related expectation values come from the same Z record.

For each shot count, repeat the energy estimate 20 times and display:

- The mean of the repeated estimates
- Their standard deviation, describing their spread
- The lowest and highest estimates
- The exact energy used as a reference

Here, the standard deviation describes <strong>how much repeated energy estimates vary for the same state</strong>.

Increasing shots generally reduces sampling fluctuations, but individual estimates need not become closer to the exact value in order. Examine the distribution of repeated results rather than comparing only one estimate at each shot count.

---

### What Should Change When the Result Is Poor?

Different causes can prevent the energy from becoming low enough.

If the circuit cannot prepare the needed state, a more precise angle search within that circuit cannot overcome its limit. If the same circuit can prepare a good state at other angles, examine the initial values or search process.

If the exact energy is sufficiently low but sampled estimates fluctuate strongly, first examine the measurement budget and the spread of the estimates.

To distinguish these causes, record <strong>the circuit, initial values, energy-evaluation method, and comparison reference</strong>, rather than one output number. The following questions revisit your results from this perspective.

---

### Your Turn

### (a) Comparing States Before and After the Circuit Change

Compare these two states:

- The state that achieves the first circuit's minimum energy, found in Problem 2
- The state prepared by the extended circuit at the angles returned in Problem 3

Use the provided verification tool to calculate exact energy and fidelity with the ground state, then complete the table.

| Case | Angles used | Exact energy $E$ | $E-E_0$ | Fidelity with the ground state |
|---|---|---:|---:|---:|
| Best state of the first circuit | … | … | … | … |
| Optimization result of the extended circuit | … | … | … | … |

Then explain the following.

1. Does each state meet this lab's energy and fidelity criteria?
2. How did the energy and state fidelity change after extending the circuit?
3. Why must these quantities be calculated separately from the optimizer's termination message?

When recording verification results, use <strong>a fresh calculation from the circuit prepared at the returned angles</strong>, rather than simply copying the optimizer's energy history.

---

### Answer 4a

Calculations, observations, and reasoning:

…

### Observation Experiment 4a

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. If the output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_4a():
    if "2bc" not in LAB_RESULTS or "3b" not in LAB_RESULTS:
        raise ValueError('Run the 2(b,c) scan and 3(b) optimization first.')
    first_theta = LAB_RESULTS["2bc"]["best"]["theta"]
    expanded = LAB_RESULTS["3b"]["parameters"]
    results, rows = [], []
    for label, parameters, circuit in (
        ('First circuit: selected from scan', (first_theta,), real_ansatz(first_theta)),
        ('Extended circuit: optimizer result', expanded, phase_ansatz(*expanded)),
    ):
        report = validate_result(circuit)
        results.append({"label": label, "parameters": parameters, **report})
        rows.append([label, parameters, report["energy"], report["delta_energy"], report["fidelity"], report["energy_pass"], report["state_pass"]])
    show_table(['Case', 'Angles', 'Exact E', "E-E0", 'Ground-state F', 'Energy criterion', 'State criterion'], rows)
    print("Use your Problem 2 answer to check whether the first row's angle is an analytic minimum.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4a", experiment_4a, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4a", None)
    print('[Waiting] 4(a): run the earlier experiments, then compare the exact results.')

### (b) Estimating the Same State's Energy with Finite Shots

Run the provided sampling experiment with the angles from Problem 3 fixed. Do not reoptimize the angles based on the measurements in this question.

Summarize the repeated results for each shot count.

| Shots per setting | Exact energy | Mean estimate | Standard deviation | Lowest estimate | Highest estimate |
|---:|---:|---:|---:|---:|---:|
| 64 | … | … | … | … | … |
| 256 | … | … | … | … | … |
| 1,024 | … | … | … | … | … |

Answer the following questions.

- How did the spread of estimates change with the shot count? Distinguish an individual result from the trend in repeated results.
- If an estimate is below $E_0$, does that record mean a state violating the variational principle was prepared? If there are no such estimates, can you conclude that such an estimate could never occur in another record?
- What is wrong with selecting only the smallest of 20 estimates at the same angles and reporting it as “the energy of this state”?

Finally, state <strong>what information should accompany a measurement-based result besides a single estimate</strong>.

---

### Answer 4b

Prediction and calculations before running:

…

Observations and explanation:

…

### Observation Experiment 4b

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. If the output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_4b():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    parameters = LAB_RESULTS["3b"]["parameters"]
    circuit = phase_ansatz(*parameters)
    reports = repeated_energy_estimates(circuit, energy_from_expectations,
                                        shots_values=(64, 256, 1024), repeats=20, seed=20261109)
    reference = ground_reference()["energy"]
    show_table(['Shots per setting', 'Exact E', 'Mean of 20 estimates', 'Sample standard deviation', 'Lowest estimate', 'Highest estimate', 'Total shots / estimate'],
               [[r["shots_per_setting"], r["exact_energy"], r["mean"], r["sample_std"], r["min"], r["max"], r["total_shots_per_estimate"]] for r in reports])
    show_table(['Estimate order', "N=64", "N=256", "N=1024"],
               [[i+1, *[r["energies"][i] for r in reports]] for i in range(20)])
    show_table(['Shots per setting', 'Estimates below E0', 'Total shots for this condition'],
               [[r["shots_per_setting"], sum(value < reference for value in r["energies"]), r["total_shots"]] for r in reports])
    print('Angles were fixed in all sampling experiments:', parameters)
    print('The three Z statistics share one measurement record. Z and XY samples are generated separately for each energy estimate.')
    return {"parameters": parameters, "reports": reports}

if RUN_OBSERVATIONS:
    _ = run_demo("4b", experiment_4b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4b", None)
    print('[Waiting] 4(b): this experiment estimates energy at fixed angles. Enable observations.')

### (c) Choosing the Next Step from a Results Report

The provided comparison keeps the extended circuit and objective unchanged and runs optimization from

$$
(0.4,0.6)
\quad\text{or}\quad
(0,0)
$$

as the initial values. Check each run's termination message, returned angles, and exact energy alongside the previous results.

Then examine the following three reports. Separate established observations from claims not yet supported, and propose what to do next.

> <strong>Report A</strong>  
> “We tried many angles with the first circuit but did not reach the ground energy. A finer angle grid and a longer optimization run will solve the problem.”

Evaluate this using the energy expression and minimum from Problem 2. Explain whether more searching can solve this particular problem.

> <strong>Report B</strong>  
> “The optimizer terminated successfully when we ran the extended circuit from one initial point. We have therefore found the lowest energy this circuit can achieve.”

Compare runs of the same circuit from different initial values. Explain what successful termination alone establishes and how the claim should change if another set of angles gives a lower energy.

> <strong>Report C</strong>  
> “Energy estimates at the same angles differed between runs. The circuit therefore seems to prepare states of different quality each time.”

Evaluate this using the state-preparation conditions and estimation method in this experiment. Distinguish a change in the state from a fluctuation in its estimated energy.

For each report, choose the necessary next steps from the list below and explain why. If several are needed, you may suggest an order.

- Examine the circuit's set of preparable states
- Modify the circuit structure
- Change the initial values or search settings
- Recalculate the exact energy
- Increase shots and repeat the estimates
- Revise how the result is reported

Do not justify your choice merely by saying “the value is poor.” Explain <strong>which observation supports the need for that action</strong>.

---

### Answer 4c

Prediction and calculations before running:

…

Observations and explanation:

…

### Observation Experiment 4c

Write your calculations and predictions first. After passing the required code checks, run this experiment and add your observations to your answer. If the output says it is waiting, check the observation setting and the earlier check cells.

In [ ]:
def experiment_4c():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    reports = [LAB_RESULTS["3b"], optimize_vqe(vqe_objective, initial=(0.0, 0.0))]
    results, rows = [], []
    for report in reports:
        validation = validate_result(phase_ansatz(*report["parameters"]))
        results.append({"optimizer": report, "validation": validation})
        rows.append([report["initial"], report["parameters"], report["success"],
                     validation["energy"], validation["delta_energy"], validation["fidelity"], report["nfev"], report["nit"]])
    show_table(['Initial values', 'Returned angles', 'Successful termination', 'Exact E', "E-E0", 'Ground-state F', 'Evaluations', 'Number of repetitions'], rows)
    for report in reports:
        print('Initial values', report["initial"], 'Termination message:', report["message"])
    print('These are actual records for the same circuit and the current BFGS settings. Do not generalize them to all optimizers.')
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4c", experiment_4c, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4c", None)
    print('[Waiting] 4(c): examine the initial-value comparison together with the earlier results.')

## Final Application — What If One Energy Term Changes Sign?

Keep the circuit structure and measurement method unchanged, but change the coefficient of the Hamiltonian's $X_0Y_1$ term.

$$
\mathcal H'
=
-Z_0Z_1+X_0Y_1-\frac12(Z_0+Z_1).
$$

Using the old angles prepares the same state. However, the Hamiltonian used to evaluate that state has changed, so its energy must be recalculated.

Before running a new optimization, write your predictions and proposed modification in this order.

1. <strong>Predict the old state's new energy.</strong>  
   Use the four expectation values obtained earlier to predict the energy under $\mathcal H'$ at the unchanged angles. Write an expression showing which term's contribution changes.

2. <strong>Propose a parameter change.</strong>  
   Suggest how to change $\theta,\phi$ while retaining the circuit structure. Explain using the relationship between the changed energy term and the state.

3. <strong>Check the prediction and modified result.</strong>  
   Pass `xy_coeff=+1.0` to your energy function to calculate the new energy. Then use the provided reference calculation to find the new Hamiltonian's ground energy and ground state, and check the modified state.

| Prepared state | Angles used | Predicted new energy | Calculated new energy | Difference from new ground energy | Fidelity with new ground state |
|---|---|---:|---:|---:|---:|
| Original angles unchanged | … | … | … | … | … |
| Proposed modified angles | … | … | … | … | … |

The comparison reference must also match the new Hamiltonian. Calculate fidelity with <strong>the ground state of $\mathcal H'$</strong>, not that of the original problem.

You do not need to write a new circuit or optimizer. Briefly summarize <strong>which tools you kept and which choices you changed when the Hamiltonian changed</strong>, and why the changes are appropriate.

## Final Application Answer

Prediction and calculations before running:

…

Observations and explanation:

…

### Checking the Application

Enter the predicted new energy of the old state in `PREDICTED_NEW_ENERGY`, and your proposed angles in `APPLICATION_PARAMETERS` in `(theta, phi)` order. Then set `RUN_APPLICATION=True` and run the experiment below. Also write the modified state's predicted energy in the written-answer table before running. These variables are experiment inputs, not new function TODOs.

In [ ]:
RUN_APPLICATION = False
APPLICATION_PARAMETERS = None
PREDICTED_NEW_ENERGY = None

In [ ]:
def application_experiment():
    if "3b" not in LAB_RESULTS:
        raise ValueError('Run the 3(b) optimization first.')
    if APPLICATION_PARAMETERS is None or PREDICTED_NEW_ENERGY is None:
        raise ValueError('First enter your proposed angles and the predicted new energy of the old state.')
    proposed = tuple(map(float, APPLICATION_PARAMETERS))
    if len(proposed) != 2 or not np.isfinite(proposed).all():
        raise ValueError('Proposed angles must be two finite values: theta and phi.')
    predicted = float(PREDICTED_NEW_ENERGY)
    if not np.isfinite(predicted):
        raise ValueError('The predicted energy must be a finite real number.')
    baseline = LAB_RESULTS["3b"]["parameters"]
    results, rows = [], []
    for label, parameters in (('Original angles unchanged', baseline), ("Student's proposed angles", proposed)):
        circuit = phase_ansatz(*parameters)
        values = expectations_from_circuit(circuit)
        energy = float(energy_from_expectations(values, xy_coeff=1.0))
        validation = validate_result(circuit, xy_coeff=1.0)
        assert np.isclose(energy, validation["energy"], atol=ATOL, rtol=0), "Check the new Hamiltonian's coefficient and the energy evaluator."
        rows.append([label, parameters, energy, validation["delta_energy"], validation["fidelity"]])
        results.append({"label": label, "parameters": parameters, "values": values, **validation})
    show_table(['Prepared state', 'Angles', 'E under new H', "E-E0'", 'New ground-state F'], rows)
    matches = bool(np.isclose(predicted, results[0]["energy"], atol=ENERGY_TOL, rtol=0))
    show_table(['Predicted new energy of old state', 'Calculated new energy', 'Matches prediction'],
               [[predicted, results[0]["energy"], matches]])
    print("Exact E0' of the new Hamiltonian:", results[0]["reference_energy"])
    print('The reference state was also recalculated for the new Hamiltonian.')
    return {"rows": results, "predicted_new_energy": predicted, "prediction_matches": matches}

LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment, CHECK_LABELS)
else:
    print('[Waiting] Final application: write your prediction and proposed angles, then set RUN_APPLICATION to True.')

## Lab 4 Wrap-Up

Check the status of all three code checks and the explanations for every subquestion. In three or four sentences, distinguish accurate energy evaluation, the states a circuit can express, the optimizer's search, and estimates from finite measurement records. Include how you would use evidence to choose the next step when a result is not good enough.

### Wrap-Up Answer

Calculations, observations, and reasoning:

…

In [ ]:
expected_checks = ['1b · Energy evaluator', '3a · Relative-phase circuit', '3b · Optimization objective']
show_table(['Code check', 'State'], [[name, CHECK_STATUS.get(name, 'Not run')] for name in expected_checks])
if all(CHECK_STATUS.get(name) == 'Passed' for name in expected_checks):
    print('All three code checks passed. Review written answers and the final application separately.')
else:
    print('Complete and run the TODO and check cells, then rerun the connected experiments.')